# GraphRAG: Social Network Analysis


#### Objectives

- GraphRAG - combining graph traversal with retrieval - augmented generation.
- Building intelligent Q & A systems over connected data.
- Practical applications with StackOverflow and Movies datasets

#### We will use

- Neo4j AuraDB free tier account
- LangChain and Groq API key (or other LLM provider)

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print("API KEY for OPENAI is not set. Please check your .env file.")
else:
    print("✓ OPENAI_API_KEY loaded successfully.")

if not os.environ.get("GROQ_API_KEY"):
    print("API key for Groq is not set. Please check your .env file.")
else:
    print("✓ API key loaded successfully.")

In [ ]:
from neo4j import GraphDatabase

# Configurare conexiune - COMPLETAȚI CU DATELE VOASTRE in .env
NEO4J_URI = os.getenv("NEO4J_URI", "neo4j+ssc://....databases.neo4j.io:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "your-password")
AUTH = (NEO4J_USERNAME, NEO4J_PASSWORD )  # Username și parola voastră

driver = GraphDatabase.driver(NEO4J_URI, auth=AUTH)

def run_query(query, parameters=None):
    """Funcție helper pentru rularea query-urilor"""
    with driver.session() as session:
        result = session.run(query, parameters or {})
        return [record.data() for record in result]

# Test conexiune
try:
    driver.verify_connectivity()
    print("✓ Conectare reușită la Neo4j!")
except Exception as e:
    print(f"✗ Eroare la conectare: {e}")

Import the sample dataset _Social Network Analysis_ from [sample datasets](https://console-preview.neo4j.io/guides/sample-datasets) step 2, from the _Social Network Analysis_ tutorial.

Test the following queries.

In [ ]:
import pandas as pd

match_user_and_question  = """
MATCH (u:User)-[a:ASKED]->(q:Question)
RETURN u,a,q
"""

result = run_query(match_user_and_question )
df = pd.DataFrame(result)
print(df)

In [ ]:
user_number_of_question  = """
MATCH (u:User)-[:ASKED]->(q:Question)
RETURN u.display_name, count(*) AS questions
"""

result = run_query(user_number_of_question)
df = pd.DataFrame(result)
print(df)

## Graph Databases for Social Networks

Traditional databases:
- Multi-hop relationships ("friends of friends who like similar content")
- Complex pattern matching
- Path finding and traversals

Graphs:
- Natural representation of connections
- Fast relationship traversal
- Flexible schema evolution

### StackOverflow Social Network - Examples

A simplified StackOverflow graph with:
- Users (developers)
- Questions
- Answers
- Tags
- Comments
- Relationships: ASKED, ANSWERED, COMMENTED, TAGGED, COMMENTED_ON, PROVIDED

### Explore the Graph with Cypher

#### Neo4jGraph
Purpose: A LangChain wrapper that provides a Python interface to interact with Neo4j graph databases, making it easy to integrate graph data into LLM applications.

Key Features:
- Query Execution: Run Cypher queries directly from Python
- Schema Introspection: Automatically extracts and understands your graph structure (node labels, relationship types, properties)
- LLM Integration: Works seamlessly with LangChain chains like GraphCypherQAChain to translate natural language to Cypher
Graph Context: Provides structured graph data as context for LLM prompts.

In [ ]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD
)

In [ ]:
# Count nodes by type
result = graph.query("""
MATCH (n)
RETURN labels(n)[0] as NodeType, count(*) as Count
ORDER BY Count DESC
""")
print("Node counts:")
for row in result:
    print(f"  {row['NodeType']}: {row['Count']}")

In [ ]:
# Find most active users
result = graph.query("""
MATCH (u:User)-[:PROVIDED]->(a:Answer)
WITH u, count(a) as answers
RETURN u.display_name as User, u.uuid as UUID, answers as Answers
ORDER BY answers DESC
""")
print("\nMost active answerers:")
for row in result:
    print(f"  {row['User']}: {row['Answers']} answers (uuid: {row['UUID']})")

In [ ]:
# Find experts on specific topics
result = graph.query("""
MATCH (u:User)-[:PROVIDED]->(a:Answer)-[:ANSWERED]->(q:Question)-[:TAGGED]->(t:Tag)
WHERE t.name IN ['graphql', 'kubernetes']
WITH u, t.name as topic, count(a) as answers, avg(a.score) as avg_score
RETURN u.display_name as Expert, topic as Topic, answers as Answers, round(avg_score, 1) as AvgVotes
ORDER BY answers DESC, avg_score DESC
""")
print("\nExperts by topic:")
for row in result:
    print(f"  {row['Expert']}: {row['Topic']} ({row['Answers']} answers, {row['AvgVotes']} avg votes)")

In [ ]:
# Add follows relationship

create_follows = """
MATCH (u1:User {user_id: 634}),
      (u2:User {user_id: 652}),
      (u3:User {user_id: 654}),
      (u4:User {user_id: 608}),
      (u5:User {user_id: 623})
CREATE (u2)-[:FOLLOWS]->(u3)
CREATE (u4)-[:FOLLOWS]->(u3)
CREATE (u4)-[:FOLLOWS]->(u5)
CREATE (u1)-[:FOLLOWS]->(u5)
CREATE (u2)-[:FOLLOWS]->(u1);
"""

graph.query(create_follows)

In [ ]:
create_questions = """
MATCH (u1:User {user_id: 634}),
      (u2:User {user_id: 652}),
      (u3:User {user_id: 654}),
      (u4:User {user_id: 608}),
      (u5:User {user_id: 623})
CREATE (t1:Tag {name: 'rag'})
CREATE (t2:Tag {name: 'vector-search'})
CREATE (t3:Tag {name: 'langchain'})
CREATE (q1:Question {id: 'q1',
                     title: 'How to implement GraphRAG with LangChain and Neo4j?',
                     body_markdown: 'I want to build a RAG system that can traverse graph relationships while retrieving context. What is the best approach using LangChain with Neo4j?',
                     view_count: 1250})
CREATE (q2:Question {id: 'q2',
                     title: 'Neo4j vector index vs traditional vector databases',
                     body_markdown: 'Should I use Neo4j vector indexes for embeddings or stick with dedicated vector databases like Pinecone? What are the tradeoffs?',
                     view_count: 890})
CREATE (q3:Question {id: 'q3',
                     title: 'Optimizing Cypher queries for large graphs',
                     body_markdown: 'My graph has 10M+ nodes and traversal queries are slow. What are best practices for indexing and query optimization?',
                     view_count: 2100})
CREATE (q4:Question {id: 'q4',
                     title: 'Combining semantic search with graph traversal',
                     body_markdown: 'How can I find semantically similar nodes and then explore their neighborhood in the graph?',
                     view_count: 650})
CREATE (a1:Answer {id: 'a1',
                   body_markdown: 'You should use GraphCypherQAChain from LangChain. It translates natural language to Cypher queries and retrieves graph context. Here is a code example...',
                   vo: 12})
CREATE (a2:Answer {id: 'a2',
                   body_markdown: 'Neo4j vector indexes are great when you need both graph relationships and semantic search. Use dedicated DBs if you only need vector similarity.',
                   view_count: 6})
CREATE (a3:Answer {id: 'a3',
                   body_markdown: 'Create indexes on frequently queried properties, use PROFILE to analyze queries, and consider graph algorithms for preprocessing.',
                   view_count: 18})
CREATE (a4:Answer {id: 'a4',
                   body_markdown: 'Use Neo4j vector index for initial similarity search, then run Cypher traversal from those nodes. This hybrid approach works well.',
                   view_count: 7})
CREATE (u2)-[:ASKED]->(q1)
CREATE (u4)-[:ASKED]->(q2)
CREATE (u1)-[:ASKED]->(q3)
CREATE (u2)-[:ASKED]->(q4)

// Users answering questions
CREATE (u3)-[:PROVIDED]->(a1)
CREATE (u3)-[:PROVIDED]->(a2)
CREATE (u5)-[:PROVIDED]->(a3)
CREATE (u3)-[:PROVIDED]->(a4)

// Answers to questions
CREATE (a1)-[:ANSWERED]->(q1)
CREATE (a2)-[:ANSWERED]->(q2)
CREATE (a3)-[:ANSWERED]->(q3)
CREATE (a4)-[:ANSWERED]->(q4)

// Question tags
CREATE (q1)-[:TAGGED]->(t1)
CREATE (q1)-[:TAGGED]->(t2)
CREATE (q1)-[:TAGGED]->(t3)
CREATE (q2)-[:TAGGED]->(t1)
CREATE (q2)-[:TAGGED]->(t2)
CREATE (q3)-[:TAGGED]->(t3)
CREATE (q3)-[:TAGGED]->(t1)
CREATE (q4)-[:TAGGED]->(t2)
CREATE (q4)-[:TAGGED]->(t3);
"""

graph.query(create_questions)

In [ ]:
# Find influential users (using PageRank-like logic)
result = graph.query("""
MATCH (u:User)<-[:FOLLOWS]-(follower:User)
WITH u, count(follower) as followers
MATCH (u)-[:PROVIDED]->(a:Answer)
WITH u, followers, avg(a.score) as avg_score, count(a) as total_answers
RETURN u.name as User,
       followers as Followers,
       total_answers as Answers,
       round(avg_score, 1) as AvgScore
ORDER BY Followers DESC, AvgScore DESC
""")
print("\nInfluential users:")
for row in result:
    print(f"  {row['User']}: {row['Followers']} followers, {row['Answers']} answers, {row['AvgScore']} avg votes")

## GraphRAG

### Traditional RAG
1. Query → Vector search → Retrieve similar documents → LLM generates answer

### GraphRAG
1. Query → Vector search OR Cypher query → Retrieve connected subgraph → LLM generates answer with graph context

### Why GraphRAG?
- **Relationship-aware**: Understands how entities connect
- **Multi-hop reasoning**: Can traverse relationships ("Alice's followers who answered questions tagged 'Neo4j'")
- **Structured context**: Graph provides explicit structure for reasoning
- **Hybrid search**: Combines semantic similarity with graph topology

### Approach 1: Cypher Query Chain (Graph-First)

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai/gpt-oss-120b", model_provider="groq")

# Refresh graph schema
graph.refresh_schema()

print("Graph Schema:")
print(graph.schema)

#### GraphCypherQAChain
Purpose: A LangChain chain that enables natural language querying of graph databases by automatically translating questions into Cypher queries, executing them, and generating natural language answers.

How It Works (3-step pipeline):
- Question → Cypher: LLM converts natural language question to Cypher query
- Execute: Runs the Cypher query against Neo4j graph
- Results → Answer: LLM synthesizes query results into a natural language response

In [ ]:
from langchain_neo4j import GraphCypherQAChain

# Create GraphCypherQAChain - translates natural language to Cypher
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    return_intermediate_steps=True,
    allow_dangerous_requests=True
)

print("✓ GraphCypherQAChain initialized")

In [ ]:
# Example 1: Simple question about users
question = "Who are the experts on graphql and how many answers have they provided?"
result = cypher_chain.invoke({"query": question})

print(f"\nQuestion: {question}")
print(f"\nGenerated Cypher:\n{result['intermediate_steps'][0]['query']}")
print(f"\nAnswer: {result['result']}")

In [ ]:
# Example 2: Multi-hop relationship query
question = "Provide the names of the users that have answered questions about 'neo4j'?"
result = cypher_chain.invoke({"query": question})

print(f"\nQuestion: {question}")
print(f"\nGenerated Cypher:\n{result['intermediate_steps'][0]['query']}")
print(f"\nAnswer: {result['result']}")

In [ ]:
# Example 3: Aggregation and pattern matching
question = "What are the most popular topics based on question comments?"
result = cypher_chain.invoke({"query": question})

print(f"\nQuestion: {question}")
print(f"\nGenerated Cypher:\n{result['intermediate_steps'][0]['query']}")
print(f"\nAnswer: {result['result']}")

### Multi-Hop Reasoning Chains

Multi-hop reasoning - traversing relationships across multiple steps to find complex patterns and connections that would be difficult with traditional RAG.

**Multi-Hop Reasoning**
- Following relationships across multiple nodes to discover indirect connections
- Example: "Find experts who are followed by people who answered questions I upvoted"
- Each "hop" is a relationship traversal in the graph
- Discovers non-obvious patterns and connections
- Captures network effects (influence, authority, community)
- Provides richer context than single-hop queries
- Enables recommendation and discovery use cases

In [ ]:
# 3-hop reasoning - Discover related topics through expert networks
question = """What topics are discussed by experts who collaborate with people
who answered questions about tag langchain?"""

result = cypher_chain.invoke({"query": question})

print(f"\nQuestion: {question}")
print(f"\nGenerated Cypher:\n{result['intermediate_steps'][0]['query']}")
print(f"\nAnswer: {result['result']}")
print("\nExplanation: 3 hops - LangChain answerers → their collaborators → topics those collaborators discuss.")

### Approach 2: Vector Search + Graph Context (Hybrid)

In [ ]:
from openai import OpenAI

model_ada = "text-embedding-ada-002"
openai_client = OpenAI()
print(openai_client.models.list())


# Define a function to generate embeddings
def get_embedding(text, model):
    """Generates vector embeddings for the given text."""

    embedding = openai_client.embeddings.create(input=[text], model=model).data[0].embedding
    return embedding

In [ ]:
# Add embeddings for question with tags rag and langchain
questions = graph.query("""
MATCH (q:Question)-[:TAGGED]->(t:Tag)
WHERE t.name IN ['rag','langchain']
RETURN q.id as id, q.title + ' ' + q.body_markdown as body_markdown
""")

for question in questions:
    if question['body_markdown']:
        embedding = get_embedding(text = question['body_markdown'], model = model_ada)

        graph.query("""
        MATCH (q:Question {id: $id})
        SET q.embedding = $embedding
        """, {"id": question['id'], "embedding": embedding})

print("✓ Question embeddings created")

In [ ]:
# Create vector index in Neo4j
try:
    graph.query("""
    CREATE VECTOR INDEX question_embeddings IF NOT EXISTS
    FOR (q:Question)
    ON q.embedding
    OPTIONS {indexConfig: {
        `vector.dimensions`: 1536,
        `vector.similarity_function`: 'cosine'
    }}
    """)
    print("✓ Vector index created")
except Exception as e:
    print(f"Note: {e}")

In [ ]:
# Hybrid search: Find similar questions and expand context with graph
def hybrid_graphrag_search(query_text, top_k=2):
    """
    1. Find semantically similar questions using vector search
    2. Expand context by traversing graph relationships
    3. Return enriched context for LLM
    """

    query_embedding = get_embedding(text = query_text, model = model_ada)

    # Vector search + graph traversal
    cypher_query = """
    CALL db.index.vector.queryNodes('question_embeddings', $top_k, $query_embedding)
    YIELD node as q, score

    // Get the question details
    MATCH (asker:User)-[:ASKED]->(q)

    // Get answers and answerers
    OPTIONAL MATCH (q)<-[:ANSWERED]-(a:Answer)<-[:PROVIDED]-(answerer:User)

    // Get tags
    OPTIONAL MATCH (q)-[:TAGGED]->(t:Tag)

    RETURN
        q.title as question_title,
        q.body_markdown as question_body,
        q.view_count as view_count,
        score as similarity_score,
        asker.display_name as asked_by,
        collect(DISTINCT {answer: a.body_markdown, score: a.score, author: answerer.display_name}) as answers,
        collect(DISTINCT t.name) as tags
    ORDER BY similarity_score DESC
    """

    results = graph.query(cypher_query, {
        "query_embedding": query_embedding,
        "top_k": top_k
    })

    return results


# Test hybrid search
query = "How do I use vectors with graph databases?"
results = hybrid_graphrag_search(query)

print(f"Query: {query}\n")
for i, result in enumerate(results, 1):
    print(f"Result {i} (similarity: {result['similarity_score']:.3f}):")
    print(f"  Question: {result['question_title']}")
    print(f"  Asked by: {result['asked_by']}")
    print(f"  Tags: {', '.join(result['tags'])}")
    print(f"  Answers: {len([a for a in result['answers'] if a['answer']])}")
    print()

In [ ]:
# Build complete GraphRAG pipeline
def graphrag_qa(question, llm_model="gpt-4"):
    """
    Complete GraphRAG pipeline:
    1. Semantic search for relevant questions
    2. Graph traversal for context
    3. LLM generation with enriched context
    """
    # Get graph context
    graph_context = hybrid_graphrag_search(question, top_k=3)

    # Format context for LLM
    context_parts = []
    for i, ctx in enumerate(graph_context, 1):
        context_text = f"""
Question {i}: {ctx['question_title']}
Asked by: {ctx['asked_by']} )
Tags: {', '.join(ctx['tags'])}
Body: {ctx['question_body']}

Answers:
"""
        for j, answer in enumerate([a for a in ctx['answers'] if a['answer']], 1):
            context_text += f"  {j}. By {answer['author']}, score: {answer['score']})\n"
            context_text += f"     {answer['answer'][:200]}...\n"

        context_parts.append(context_text)

    full_context = "\n---\n".join(context_parts)

    # Create prompt
    prompt = f"""
You are a helpful assistant answering questions about programming and technology.
Use the following context from a StackOverflow-like community to answer the question.
Consider the reputation of users and votes on answers when determining credibility.

Context:
{full_context}

Question: {question}

Provide a comprehensive answer based on the context. Mention which experts or answers you're drawing from.
"""

    # Get LLM response
    llm = init_chat_model("openai/gpt-oss-120b", model_provider="groq")
    response = llm.invoke(prompt)

    return {
        "answer": response.content,
        "context": graph_context
    }

print("✓ GraphRAG pipeline ready")

In [ ]:
# Test the complete pipeline
question = "What are best practices for using Neo4j with vector embeddings in a RAG system?"
result = graphrag_qa(question)

print(f"Question: {question}\n")
print(f"Answer:\n{result['answer']}")

In [ ]:
question = "Who should I follow if I want to learn about GraphRAG and LangChain?"
result = graphrag_qa(question)

print(f"Question: {question}\n")
print(f"Answer:\n{result['answer']}")

#### Exercise 1: Graph Exploration

To load the Movies dataset:

1. Open your [Neo4j Browser](https://browser.neo4j.io/) at your AuraDB instance URL
2. Run: :play movies
3. Follow the guide and execute the Cypher queries

Write Cypher queries to explore the Movies graph:

1. Find all movies Tom Hanks acted in
2. Find all actors who acted in "The Matrix"
3. Find actors who have worked with Tom Hanks (co-actors)
4. Find the shortest path between Kevin Bacon and Meg Ryan

#### Exercise 2: Add Vector Embeddings

1. Add embeddings to movie taglines and plots
2. Create a vector index
3. Implement semantic search for similar movies

#### Exercise 3: Build Movie GraphRAG

1. Finds semantically similar movies
2. Includes cast/crew connections
3. Generates natural language recommendations

#### Test with different queries:
find_similar_movies("artificial intelligence and humanity")
movie_recommendation_qa("Recommend action movies with Tom Cruise")
movie_recommendation_qa("Movies similar to The Matrix but older")
etc.